# Columbia River Gorge Hiking Trails Analysis

**Dataset**: HikingTrails_TheGorge.csv  
**Purpose**: Analyzing trail characteristics, difficulty ratings, and family-friendliness patterns

## Research Questions:
1. Which trail characteristics (distance, elevation gain, and highest point) are most strongly associated with higher difficulty ratings?
2. How do trail features such as elevation gain and distance differ between family-friendly and non-family-friendly trails?
3. Do seasonal accessibility patterns cluster around certain difficulty levels or elevation thresholds?

In [1]:
import pandas as pd
import numpy as np

# Load the hiking trails dataset
df = pd.read_csv('HikingTrails_TheGorge.csv')

## Initial Data Exploration

Before analyzing, I need to understand the structure and quality of the data.

In [2]:
# OPERATION 1: df.head() and df.info() - Understanding the data structure
# Question: What does the dataset look like and what are the data types?
# Why this matters: I need to see if numeric columns are stored as strings (with 'feet', 'miles', commas)
# and identify which columns need cleaning before analysis

print("First 5 rows of the dataset:")
print(df.head())
print("\n" + "="*80 + "\n")

print("Dataset structure and data types:")
print(df.info())
print("\n" + "="*80 + "\n")

# What this tells me: The elevation and distance columns likely contain text ('feet', 'miles') 
# rather than pure numbers, which means I'll need to clean them before doing numerical analysis.
# This is common in real-world datasets scraped from websites or user inputs.

First 5 rows of the dataset:
                            Trail Name     Trail Type               Distance  \
0                  Ainsworth Loop Hike           Loop              0.5 miles   
1                   Aldrich Butte Hike   Out and Back  13.8 miles round trip   
2  Aldrich Butte-Cedar Falls Loop Hike  Lollipop loop  16.4 miles round trip   
3                     Angels Rest Hike   Out and Back   4.8 miles round trip   
4    Angels Rest-Devils Rest Loop Hike           Loop             10.8 miles   

   High Point Elevation Gain Difficulty     Seasons Family Friendly  \
0    150 feet        85 feet       Easy    All year             Yes   
1         NaN      2405 feet   Moderate  All Season              No   
2  1,140 feet      3105 feet  Difficult  Year round              No   
3   1640 feet      1475 feet   Moderate  All Season             Yes   
4   2435 feet      3040 feet   Moderate  All Season             Yes   

  Backpackable Crowded  
0           No      No  
1           N

In [3]:
# OPERATION 2: df.isnull().sum() - Finding missing data
# Question: Which columns have missing values and how many?
# Why this matters: Missing elevation or distance data could skew my analysis of trail difficulty.
# I need to know if I should filter out incomplete records or fill them with reasonable defaults.

print("Missing values per column:")
print(df.isnull().sum())
print("\n" + "="*80 + "\n")

# What this tells me: If High Point or Elevation Gain have many missing values,
# it means some trail data is incomplete. I'll need to decide whether to:
# a) exclude those trails from certain analyses
# b) investigate if there's a pattern (e.g., are easy trails less likely to report elevation?)

Missing values per column:
Trail Name          0
Trail Type          2
Distance            0
High Point         21
Elevation Gain      1
Difficulty          1
Seasons             0
Family Friendly     2
Backpackable        1
Crowded             1
dtype: int64




## Data Cleaning

Real-world data often needs cleaning. I'll convert text-based measurements to numeric values.

In [4]:
# Helper function to clean numeric columns that contain text like '1,234 feet' or '5.6 miles'
def extract_number(value):
    """
    Extracts numeric value from strings containing numbers with units.
    Example: '1,234 feet' -> 1234.0
    Example: '5.6 miles' -> 5.6
    """
    if pd.isna(value):
        return np.nan
    # Convert to string, remove commas, extract first number
    value_str = str(value).replace(',', '')
    try:
        # Split on space and take first part (the number)
        return float(value_str.split()[0])
    except:
        return np.nan

# Clean the numeric columns
df['Distance_Numeric'] = df['Distance'].apply(extract_number)
df['High_Point_Numeric'] = df['High Point'].apply(extract_number)
df['Elevation_Gain_Numeric'] = df['Elevation Gain'].apply(extract_number)

print("Cleaned columns created. Sample of cleaned data:")
print(df[['Trail Name', 'Distance', 'Distance_Numeric', 'Elevation Gain', 'Elevation_Gain_Numeric']].head())
print("\n" + "="*80 + "\n")

Cleaned columns created. Sample of cleaned data:
                            Trail Name               Distance  \
0                  Ainsworth Loop Hike              0.5 miles   
1                   Aldrich Butte Hike  13.8 miles round trip   
2  Aldrich Butte-Cedar Falls Loop Hike  16.4 miles round trip   
3                     Angels Rest Hike   4.8 miles round trip   
4    Angels Rest-Devils Rest Loop Hike             10.8 miles   

   Distance_Numeric Elevation Gain  Elevation_Gain_Numeric  
0               0.5        85 feet                    85.0  
1              13.8      2405 feet                  2405.0  
2              16.4      3105 feet                  3105.0  
3               4.8      1475 feet                  1475.0  
4              10.8      3040 feet                  3040.0  




## Question 1: Trail Characteristics and Difficulty Ratings

Which trail characteristics (distance, elevation gain, and highest point) are most strongly associated with higher difficulty ratings?

In [5]:
# OPERATION 3: df['column'].value_counts() - Understanding difficulty distribution
# Question: What are the most common difficulty levels in the dataset?
# Why this matters: If 90% of trails are 'Easy', then patterns for 'Difficult' trails 
# might be based on very few examples. This helps me understand if I have enough data
# for each difficulty category to make meaningful comparisons.

print("Distribution of trail difficulties:")
print(df['Difficulty'].value_counts())
print("\n" + "="*80 + "\n")

# What this tells me: The balance between Easy, Moderate, and Difficult trails.
# If there are very few 'Difficult' trails, I should be cautious about generalizing
# patterns. If the distribution is balanced, I can more confidently compare characteristics.

Distribution of trail difficulties:
Difficulty
Moderate                                              65
Easy                                                  62
Difficult                                             36
Difficult (scramble, exposure)                         2
Difficult due to elevation gain                        2
Moderate (in sections)                                 1
Difficult due to elevation gain and length             1
Difficult (due to climbing over logs and deep wadi     1
Difficult (steepness, length and creek crossing)       1
Name: count, dtype: int64




In [6]:
# OPERATION 4: df.groupby('column')['other'].mean() - Average characteristics by difficulty
# Question: What are the average distance, elevation gain, and high point for each difficulty level?
# Why this matters: This reveals the relationship between trail features and difficulty ratings.
# If 'Difficult' trails average 3000 feet of elevation gain while 'Easy' trails average 300 feet,
# that tells me elevation gain is a strong indicator of difficulty - which is valuable for
# hikers planning trips based on their fitness level.

print("Average trail characteristics by difficulty level:")
difficulty_stats = df.groupby('Difficulty')[['Distance_Numeric', 'Elevation_Gain_Numeric', 'High_Point_Numeric']].mean()
print(difficulty_stats.round(2))
print("\n" + "="*80 + "\n")

# What this tells me: 
# - If elevation gain increases dramatically from Easy → Moderate → Difficult,
#   it's the strongest predictor of difficulty
# - If distance doesn't increase much, it suggests difficulty is more about vertical gain than horizontal distance
# - High point might reveal whether difficult trails go to higher elevations or just have steeper climbs

Average trail characteristics by difficulty level:
                                                    Distance_Numeric  \
Difficulty                                                             
Difficult                                                      15.46   
Difficult (due to climbing over logs and deep wadi              1.00   
Difficult (scramble, exposure)                                  6.00   
Difficult (steepness, length and creek crossing)               10.00   
Difficult due to elevation gain                                13.65   
Difficult due to elevation gain and length                     14.10   
Easy                                                            2.74   
Moderate                                                        6.72   
Moderate (in sections)                                         34.70   

                                                    Elevation_Gain_Numeric  \
Difficulty                                                                   


In [7]:
# Additional analysis: Count how many trails in each difficulty have complete data
# Question: Do I have enough data points with all three characteristics for each difficulty level?

print("Number of trails with complete data for each difficulty:")
complete_data = df.dropna(subset=['Distance_Numeric', 'Elevation_Gain_Numeric', 'High_Point_Numeric'])
print(complete_data['Difficulty'].value_counts())
print("\n" + "="*80 + "\n")

# What this tells me: Whether my analysis is based on a representative sample.
# If most 'Difficult' trails are missing high point data, the averages above might be misleading.

Number of trails with complete data for each difficulty:
Difficulty
Easy                               61
Moderate                           60
Difficult                          26
Moderate (in sections)              1
Difficult due to elevation gain     1
Name: count, dtype: int64




## Question 2: Family-Friendly vs Non-Family-Friendly Trails

How do trail features such as elevation gain and distance differ between family-friendly and non-family-friendly trails?

In [8]:
# First, understand the distribution of family-friendly trails
# Question: How many trails are marked as family-friendly vs not?

print("Family-friendly trail distribution:")
print(df['Family Friendly'].value_counts())
print("\n" + "="*80 + "\n")

# What this tells me: The proportion of trails suitable for families.
# If only 10% are family-friendly, it tells me the Gorge has mostly challenging terrain.
# This is useful context for families planning trips.

Family-friendly trail distribution:
Family Friendly
Yes                                   81
No                                    63
Yes, for older kids                    8
Too long                               3
Only on easier trails                  1
Yes: but there are steep drop-offs     1
No, too long                           1
O.K. for older kids                    1
For older, adventurous kids            1
For adventurous scramblers only        1
Only short distances                   1
Yes, for a shorter distance            1
Yes (as a car shuttle)                 1
Yes, but take care at the cliffs       1
No. Stony scramble & poison oak        1
Yes, in short sections                 1
Yes, for older children                1
For older kids                         1
For a short distance                   1
Name: count, dtype: int64




In [9]:
# OPERATION 5: df.groupby() with multiple aggregations - Compare trail characteristics
# Question: What are the average and median distance and elevation gain for family-friendly vs non-family trails?
# Why this matters: This directly answers whether family-friendly trails are actually shorter and less steep.
# Using both mean and median helps me see if there are outliers (a few very long "family-friendly" trails
# would raise the mean but not the median). This is practical information for families deciding which trails to attempt.

print("Trail characteristics: Family-friendly vs Non-family-friendly:")
family_comparison = df.groupby('Family Friendly')[['Distance_Numeric', 'Elevation_Gain_Numeric']].agg(['mean', 'median', 'count'])
print(family_comparison.round(2))
print("\n" + "="*80 + "\n")

# What this tells me:
# - If family trails average 2 miles vs 8 miles for non-family, distance is a key factor
# - If elevation gain differs by 2000+ feet, that's more important than distance for families
# - The count tells me how many trails I'm averaging - important for statistical confidence
# - Median vs mean comparison reveals if there are extreme outliers in either category

Trail characteristics: Family-friendly vs Non-family-friendly:
                                   Distance_Numeric               \
                                               mean median count   
Family Friendly                                                    
For a short distance                          22.20  22.20     1   
For adventurous scramblers only                0.50   0.50     1   
For older kids                                 5.80   5.80     1   
For older, adventurous kids                    4.80   4.80     1   
No                                            11.51  11.10    63   
No, too long                                  14.40  14.40     1   
No. Stony scramble & poison oak                6.70   6.70     1   
O.K. for older kids                            5.10   5.10     1   
Only on easier trails                          1.70   1.70     1   
Only short distances                           7.10   7.10     1   
Too long                                      16.53  

In [10]:
# OPERATION 6: df[df['column'] > value] - Filter to examine specific cases
# Question: Are there any family-friendly trails with unusually high elevation gain?
# Why this matters: This finds potential anomalies - trails marked family-friendly but with
# steep climbs might be data errors, or they might be gradual climbs that are actually
# suitable for families. Either way, these outliers deserve attention. For a hiking app,
# these would be trails to flag for review or special notation.

print("Family-friendly trails with elevation gain > 1000 feet:")
family_steep = df[(df['Family Friendly'] == 'Yes') & (df['Elevation_Gain_Numeric'] > 1000)]
print(family_steep[['Trail Name', 'Distance_Numeric', 'Elevation_Gain_Numeric', 'Difficulty']].sort_values('Elevation_Gain_Numeric', ascending=False))
print(f"\nFound {len(family_steep)} family-friendly trails with >1000 ft elevation gain")
print("\n" + "="*80 + "\n")

# What this tells me: Whether "family-friendly" has exceptions or if it's a consistent category.
# If I find family trails with 2000+ feet gain, I should investigate - maybe they're rated for
# 'older kids' (which I see in the data) or maybe the data is inconsistent. This is the kind of
# data quality check that matters in real-world analysis.

Family-friendly trails with elevation gain > 1000 feet:
                                    Trail Name  Distance_Numeric  \
123                     Ruckel Ridge Loop Hike               9.0   
4            Angels Rest-Devils Rest Loop Hike              10.8   
151                       Tracy Hill Loop Hike               6.1   
100               Multnomah-Wahkeena Loop Hike               4.9   
48                       Fairy Falls Loop Hike               3.4   
3                             Angels Rest Hike               4.8   
97                               Mud Lake Hike               7.4   
76             Larch Mountain Crater Loop Hike               6.3   
60                 Herman Creek Pinnacles Hike               4.6   
169                         Wind Mountain Hike               2.9   
17                    Cape Horn Overlooks Hike               5.2   
95   Mount Defiance from Wahtum Lake Road Hike               3.2   
29                Chinidere Mountain Loop Hike              

## Question 3: Seasonal Accessibility and Difficulty/Elevation Patterns

Do seasonal accessibility patterns cluster around certain difficulty levels or elevation thresholds?

In [11]:
# First, understand seasonal patterns in the data
# Question: What are the most common seasonal accessibility patterns?

print("Most common seasonal access patterns:")
print(df['Seasons'].value_counts().head(10))
print("\n" + "="*80 + "\n")

# What this tells me: Which season descriptions are most common.
# If 'All year' dominates, most trails are low-elevation and snow-free.
# If 'Summer/Fall' is common, many trails are higher elevation with snow in winter/spring.
# This helps hikers understand seasonal constraints in the Columbia Gorge.

Most common seasonal access patterns:
Seasons
All year                                  45
Year round                                24
Summer into Fall                           7
Apr-Oct                                    6
Spring through fall                        6
Year-round except during winter storms     5
Apr-Nov                                    4
Summer into fall                           3
Year-round                                 3
All Season                                 3
Name: count, dtype: int64




In [12]:
# Create a simplified seasonal category to make analysis clearer
# Question: Can I group seasonal patterns into broader categories?

def categorize_season(season_text):
    """
    Categorizes trails into year-round vs seasonal access.
    Year-round = accessible all year (low elevation, no snow issues)
    Seasonal = limited access (high elevation, snow, or other restrictions)
    """
    if pd.isna(season_text):
        return 'Unknown'
    season_lower = str(season_text).lower()
    if 'all year' in season_lower or 'year round' in season_lower or 'year-round' in season_lower:
        return 'Year-Round'
    else:
        return 'Seasonal'

df['Season_Category'] = df['Seasons'].apply(categorize_season)

print("Distribution of year-round vs seasonal trails:")
print(df['Season_Category'].value_counts())
print("\n" + "="*80 + "\n")

Distribution of year-round vs seasonal trails:
Season_Category
Year-Round    102
Seasonal       70
Name: count, dtype: int64




In [13]:
# OPERATION 7: df.groupby() with multiple columns - Cross-tabulation of difficulty and seasonality
# Question: How does seasonal accessibility relate to trail difficulty?
# Why this matters: This reveals whether difficult trails are also more likely to be seasonal
# (possibly due to higher elevation and snow). If 'Difficult' trails are mostly seasonal,
# it tells me that elevation (which causes seasonal closures) and difficulty are related.
# This is practical knowledge for trip planning - difficult hikes might only be accessible
# in summer months.

print("Trail count by Difficulty and Seasonal Access:")
season_difficulty = pd.crosstab(df['Difficulty'], df['Season_Category'], margins=True)
print(season_difficulty)
print("\n" + "="*80 + "\n")

# What this tells me:
# - If most 'Easy' trails are year-round, they're likely at lower elevations
# - If most 'Difficult' trails are seasonal, they're likely at higher elevations with snow
# - This pattern helps predict accessibility based on difficulty rating

Trail count by Difficulty and Seasonal Access:
Season_Category                                     Seasonal  Year-Round  All
Difficulty                                                                   
Difficult                                                 32           4   36
Difficult (due to climbing over logs and deep wadi         1           0    1
Difficult (scramble, exposure)                             0           2    2
Difficult (steepness, length and creek crossing)           1           0    1
Difficult due to elevation gain                            2           0    2
Difficult due to elevation gain and length                 1           0    1
Easy                                                       9          53   62
Moderate                                                  24          41   65
Moderate (in sections)                                     0           1    1
All                                                       70         101  171




In [14]:
# Compare elevation characteristics between year-round and seasonal trails
# Question: What are the average high points and elevation gains for year-round vs seasonal trails?
# Why this matters: This tests my hypothesis that seasonal trails are at higher elevations.
# If seasonal trails average 3000+ feet high point vs 500 feet for year-round trails,
# it confirms that elevation (and resulting snow) drives seasonal closures. This helps
# hikers understand which elevations are accessible in different seasons.

print("Average elevation characteristics by seasonal access:")
season_elevation = df.groupby('Season_Category')[['High_Point_Numeric', 'Elevation_Gain_Numeric']].mean()
print(season_elevation.round(2))
print("\n" + "="*80 + "\n")

# What this tells me:
# - A large difference in high point (e.g., 3500 ft vs 800 ft) confirms elevation drives seasonality
# - If elevation gain is similar but high point differs, it means seasonal trails START higher
# - This information helps predict which trails will be accessible in winter vs summer

Average elevation characteristics by seasonal access:
                 High_Point_Numeric  Elevation_Gain_Numeric
Season_Category                                            
Seasonal                    3321.34                 2773.72
Year-Round                   943.17                  957.35




In [15]:
# OPERATION 8: df[df['column'] > value] - Identify high-elevation year-round trails
# Question: Are there any year-round accessible trails with high elevation?
# Why this matters: These are exceptions to the pattern - trails that stay accessible
# despite high elevation. They might be south-facing (more sun = less snow),
# or the data might be wrong. Either way, for a hiking recommendation system,
# these are special cases worth highlighting - high elevation winter hikes are valuable!

print("Year-round trails with high point > 2000 feet:")
high_yearround = df[(df['Season_Category'] == 'Year-Round') & (df['High_Point_Numeric'] > 2000)]
print(high_yearround[['Trail Name', 'High_Point_Numeric', 'Elevation_Gain_Numeric', 'Difficulty']].sort_values('High_Point_Numeric', ascending=False))
print(f"\nFound {len(high_yearround)} year-round trails above 2000 feet")
print("\n" + "="*80 + "\n")

# What this tells me: Whether high elevation always means seasonal closure, or if there are
# exceptions. These exceptions are valuable for winter hiking - they offer elevation and views
# without requiring snowshoes or dealing with closures. For a recommendation engine,
# these would be highlighted for winter hiking.

Year-round trails with high point > 2000 feet:
                                Trail Name  High_Point_Numeric  \
137                     Stacker Butte Hike              3220.0   
31                     Cook Hill Loop Hike              3015.0   
37                       Dog Mountain Hike              2948.0   
38                  Dog Mountain Loop Hike              2948.0   
69                  Indian Point Loop Hike              2935.0   
59                       Herman Creek Hike              2925.0   
99   Multnomah Falls-Devils Rest Loop Hike              2445.0   
36           Devils Rest via Wahkeena Hike              2435.0   
154            Upper Hardy Creek Loop Hike              2420.0   

     Elevation_Gain_Numeric Difficulty  
137                  1145.0   Moderate  
31                   2930.0   Moderate  
37                   2800.0   Moderate  
38                   2800.0   Moderate  
69                   2730.0   Moderate  
59                   2970.0  Difficult  
99   

## Summary of Findings

### Question 1: Trail Characteristics and Difficulty
The groupby analysis reveals which factors (distance, elevation gain, high point) correlate most strongly with difficulty ratings. This helps hikers understand what makes a trail "Difficult" vs "Easy."

### Question 2: Family-Friendly Trail Features  
Comparing average distance and elevation gain between family-friendly and non-family trails shows the practical thresholds that make trails suitable for families. The filtering analysis identifies outliers that might need data review.

### Question 3: Seasonal Patterns and Elevation
Cross-tabulation of difficulty and seasonality, combined with elevation analysis, reveals whether high elevation (and resulting snow) creates seasonal closures. This helps predict trail accessibility by season.

---

**Real-world value**: This analysis could inform:
- A hiking recommendation app that suggests trails based on fitness level and season
- Data quality improvements (identifying trails with inconsistent ratings)
- User education (helping hikers understand what difficulty ratings actually mean in terms of distance and elevation)